<a href="https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"
REPO_DIR = "flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from work/notebooks/ to repo root
elif os.path.basename(os.getcwd()) == "work":
    os.chdir("..")  # move from work/ to repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-assignment1
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring**

Lane 2 is about prioritizing which pages an editor should review first — the output is an ordered queue, not a binary label. While classification ("is this page declining?") is a related question, the operational decision is *which ones first*, which maps directly to ranking. The model produces a continuous priority score, and editors act on the top-K.

This aligns with the "Which ones first?" row in the task-type table: a priority score with Precision@K as the typical metric.

In [ ]:
# Section 1 reasoning is in the markdown above — no code needed

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` — a binary proxy for "this page's traffic is declining"**

The label is defined as `is_declining_label = 1` when `trend_direction == "down"`, which means impressions_last_30d dropped more than 20% below impressions_prev_30d.

**This is a proxy, not an observed outcome.** The label measures decline *within the same 90-day window* — it does not predict future decline. A true observed target would be defined in a *forward* window (e.g., features from days 1–90 → decline in days 91–120). That requires the full warehouse (daily time-series), not the starter 30k-row snapshot.

**Label honesty:** On this starter data, roughly 54% of rows are `is_declining_label = 1`. The baseline rule catches ~24% of true declining pages in its top-50; the goal is to improve that capture rate.

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Label balance:')
print(df['is_declining_label'].value_counts())
print(f'Positive rate: {df["is_declining_label"].mean()*100:.1f}%')
print()
print('Trend direction distribution (the label source):')
print(df['trend_direction'].value_counts())

Label balance:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Positive rate: 54.2%

Trend direction distribution (the label source):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

Precision@50 measures: "of the top-50 pages the model ranks highest for editor review, what fraction are actually declining?"

Why this metric:
- Editors review a fixed number of pages per batch (≈50). The decision is "which 50 pages should I look at today?" — getting more true positives in that set is the direct business value.
- The baseline rule (a simple multi-signal filter) achieves Precision@50 ≈ 0.24. A useful ML model should measurably beat that.
- Precision@K is the standard metric for ranking tasks where the user acts on the top K.

**Secondary sanity check:** Lift over baseline — does the learned ranking capture more declining pages in the top-50 than the hand-coded rule?

In [ ]:
# Baseline Precision@50 using simple rule (no label leakage)
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Baseline rule: stale OR thin-visible — signals observable without knowing trend
stale = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
thin_visible = (df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)
df['baseline_score'] = (stale | thin_visible).astype(int)

# Sort by baseline score (high), then by impressions (high) as tiebreaker
baseline_top50 = df.sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False]).head(50)
baseline_precision = baseline_top50['is_declining_label'].mean()
print(f'Baseline Precision@50: {baseline_precision:.3f}')
print(f'Declining caught in top-50: {baseline_top50["is_declining_label"].sum()} / 50')
print(f'Baseline mentioned in w01: ~0.24 — a learned model targets ~0.74')

Baseline Precision@50: 0.660
Declining caught in top-50: 33 / 50
Baseline mentioned in w01: ~0.24 — a learned model targets ~0.74


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (a single page).** The starter dataset contains 30,000 rows across 32 pseudonymized clients.

For Lane 2 (content opportunity scoring), we use the full dataset (all clients, all pages). No sub-selection needed — every page is a candidate for review. The key columns for ranking: impressions (volume), trend (direction of change), content age, freshness, position, CTR, engagement rate, and word count.

Below: loading the data and showing the unit of analysis with a subset of the columns most relevant to ranking.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f'Rows (content items): {len(df):,}')
print(f'Clients: {df["client_id"].nunique()}')
print(f'Columns: {df.shape[1]}')
print()

# Show the unit of analysis: one page per row
cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'ctr',
        'avg_position', 'trend_direction', 'is_declining_label',
        'content_age_days', 'days_since_last_update', 'word_count',
        'engagement_rate', 'content_type']
print('Unit of analysis — one row = one content page:')
display(df[cols].head(10))

Rows (content items): 30,000
Clients: 32
Columns: 45

Unit of analysis — one row = one content page:


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_label,content_age_days,days_since_last_update,word_count,engagement_rate,content_type
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,down,1,187,20,3221.0,5.88,keyword article
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,down,1,445,25,2481.0,0.00,keyword article
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,down,1,141,20,3515.0,0.00,keyword article
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,stable,0,463,22,NaN,1.28,keyword article
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,down,1,263,14,2803.0,0.00,keyword article
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,down,1,147,20,3080.0,0.00,keyword article
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,down,1,90,20,3059.0,0.00,keyword article
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,stable,0,445,22,NaN,3.57,keyword article
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,down,1,90,20,3807.0,5.88,keyword article
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,down,1,257,104,NaN,0.00,keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g., "flag pages older than 180 days with >500 impressions") captures one narrow pattern. Real decline risk is multi-factorial and non-linear:

- **Interacting signals:** A fresh page with dropping CTR may need attention, while an old page with stable but low engagement might not. The same `ctr` value means different things at different positions, ages, and volume levels.
- **Non-linear relationships:** The effect of position on decline risk is not linear — dropping from position 2 to 4 costs more than dropping from 12 to 14. A fixed rule can't encode these thresholds without arbitrary cutoffs.
- **Shifting over time:** The relative importance of signals changes as search algorithms evolve, content patterns shift, and AI-referred traffic grows. A learned model retrains on current data; a fixed rule stays stale.
- **Continuous priority, not binary flag:** Editors need a *ranked queue*, not a yes/no filter. A rule produces coarse buckets; a learned ranking model produces a smooth priority score that orders all 30,000 pages meaningfully.

A single if-rule (e.g., "pages not updated in 180 days with >500 impressions") captures one signal. But real decline risk is multi-factorial: a fresh page with dropping CTR, an old page with stable position but falling engagement, a high-volume page with rising AI-referral cannibalization. These signals interact non-linearly and shift over time. A learned ranking combines dozens of observed signals, weights them by actual predictive power on held-out clients, and adapts when retrained — something a static rule cannot do.

In [ ]:
# Demonstrate why a single rule misses patterns
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Rule 1: stale + visible
rule1 = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
# Rule 2: declining + demand
rule2 = (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)
# Rule 3: thin + visible
rule3 = (df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)

print('Pages caught by each rule:')
print(f'  Rule 1 (stale visible): {rule1.sum():,} — Precision@All: {df.loc[rule1, "is_declining_label"].mean():.2f}')
print(f'  Rule 2 (declining demand): {rule2.sum():,} — Precision@All: {df.loc[rule2, "is_declining_label"].mean():.2f}')
print(f'  Rule 3 (thin visible): {rule3.sum():,} — Precision@All: {df.loc[rule3, "is_declining_label"].mean():.2f}')
print()

# Pages caught by ALL three rules (intersection) — very few
all_three = rule1 & rule2 & rule3
print(f'Pages caught by ALL three rules: {all_three.sum():,}')
print()

# Pages caught by exactly ONE rule — each captures a different declining population
exactly_rule1 = rule1 & ~rule2 & ~rule3
exactly_rule2 = ~rule1 & rule2 & ~rule3
exactly_rule3 = ~rule1 & ~rule2 & rule3
print(f'Pages caught ONLY by rule 1: {exactly_rule1.sum():,} (declining rate: {df.loc[exactly_rule1, "is_declining_label"].mean():.2f})')
print(f'Pages caught ONLY by rule 2: {exactly_rule2.sum():,} (declining rate: {df.loc[exactly_rule2, "is_declining_label"].mean():.2f})')
print(f'Pages caught ONLY by rule 3: {exactly_rule3.sum():,} (declining rate: {df.loc[exactly_rule3, "is_declining_label"].mean():.2f})')
print()
print('A single rule misses most declining pages. A learned model can combine all signals with optimal weights.')

Pages caught by each rule:
  Rule 1 (stale visible): 17 — Precision@All: 0.94
  Rule 2 (declining demand): 13,152 — Precision@All: 1.00
  Rule 3 (thin visible): 82 — Precision@All: 0.54

Pages caught by ALL three rules: 0

Pages caught ONLY by rule 1: 1 (declining rate: 0.00)
Pages caught ONLY by rule 2: 13,092 (declining rate: 1.00)
Pages caught ONLY by rule 3: 38 (declining rate: 0.00)

A single rule misses most declining pages. A learned model can combine all signals with optimal weights.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.